# 🍎 YOLOv8 — Détection de Fruits Personnalisée

**Notebook Google Colab — Entraînement Professionnel**

---

### 📋 Classes détectées
| ID | Classe | ID | Classe |
|----|--------|----|--------|
| 80 | melon | 85 | tomato |
| 81 | grape | 86 | strawberry |
| 82 | peach | 87 | mango |
| 83 | avocado | 88 | pear |
| 84 | pineapple | | |

### 🗺️ Plan du notebook
1. **Setup** — Montage Drive, dépendances, GPU
2. **Dataset** — Organisation, split train/val, YAML
3. **Vérification** — Intégrité des données
4. **Training** — YOLOv8 + augmentation + callbacks
5. **Évaluation** — Métriques + visualisations
6. **Export** — ONNX + sauvegarde Drive

> ⚠️ **Prérequis** : Activer le GPU dans `Exécution > Modifier le type d'exécution > GPU (T4)`

---
## 🔧 Cellule 0 — Configuration globale
**Modifier ces paramètres selon votre setup Drive.**

In [ ]:
# ============================================================
# CONFIGURATION GLOBALE — À MODIFIER SELON VOTRE PROJET
# ============================================================

# --- Chemins Google Drive ---
DRIVE_BASE       = '/content/drive/MyDrive'
RAW_IMAGES_DIR   = f'{DRIVE_BASE}/fruits_dataset/images'   # Dossier contenant vos images
RAW_LABELS_DIR   = f'{DRIVE_BASE}/fruits_dataset/labels'   # Dossier contenant vos labels .txt
DRIVE_OUTPUT_DIR = f'{DRIVE_BASE}/yolov8_fruits_runs'      # Sauvegarde des résultats

# --- Paramètres Dataset ---
VAL_SPLIT   = 0.20    # 20% validation
RANDOM_SEED = 42      # Reproductibilité

# --- Classes (ID YOLO → Nom) ---
# ⚠️  Les IDs dans vos .txt doivent correspondre à l'index 0-based ci-dessous
#     id 0 dans le .txt = melon, id 1 = grape, etc.
CLASS_NAMES = [
    'melon',       # 0  (original COCO id: 80)
    'grape',       # 1  (original COCO id: 81)
    'peach',       # 2  (original COCO id: 82)
    'avocado',     # 3  (original COCO id: 83)
    'pineapple',   # 4  (original COCO id: 84)
    'tomato',      # 5  (original COCO id: 85)
    'strawberry',  # 6  (original COCO id: 86)
    'mango',       # 7  (original COCO id: 87)
    'pear',        # 8  (original COCO id: 88)
]
NUM_CLASSES = len(CLASS_NAMES)

# --- Hyperparamètres d'entraînement ---
MODEL_SIZE   = 'yolov8n'   # n=nano, s=small, m=medium (nano recommandé pour ~100 imgs/classe)
EPOCHS       = 150         # Avec early stopping, s'arrêtera avant si nécessaire
BATCH_SIZE   = 16          # Adapter si OOM: essayer 8
IMG_SIZE     = 640         # Résolution d'entraînement
PATIENCE     = 30          # Early stopping: arrêt si pas d'amélioration sur N epochs
LR0          = 0.01        # Learning rate initial
LRF          = 0.001       # Learning rate final (scheduler cosine)
WEIGHT_DECAY = 0.0005      # Régularisation L2
OPTIMIZER    = 'AdamW'     # AdamW recommandé pour petit dataset
WARMUP_EPOCHS= 3           # Warmup learning rate

# --- Augmentation ---
AUGMENT_PARAMS = dict(
    hsv_h      = 0.015,   # Teinte
    hsv_s      = 0.7,     # Saturation
    hsv_v      = 0.4,     # Valeur/Luminosité
    degrees    = 15.0,    # Rotation ±15°
    translate  = 0.1,     # Translation
    scale      = 0.5,     # Zoom ±50%
    shear      = 5.0,     # Cisaillement
    perspective= 0.0005,  # Perspective
    flipud     = 0.3,     # Flip vertical (30% de chance)
    fliplr     = 0.5,     # Flip horizontal (50% de chance)
    mosaic     = 1.0,     # Mosaïque (très efficace pour petits datasets)
    mixup      = 0.1,     # MixUp léger
    copy_paste = 0.1,     # Copy-Paste augmentation
)

# --- Options avancées ---
USE_MIXED_PRECISION = True   # float16 pour accélérer l'entraînement
CACHE_DATASET       = True   # Mettre en cache RAM (plus rapide si RAM suffisante)
RESUME_TRAINING     = False  # Reprendre depuis un checkpoint?
RESUME_CHECKPOINT   = ''     # Chemin vers le checkpoint (si RESUME_TRAINING=True)

print('✅ Configuration chargée avec succès')
print(f'   Classes: {NUM_CLASSES} → {CLASS_NAMES}')
print(f'   Modèle:  {MODEL_SIZE}.pt')
print(f'   Epochs:  {EPOCHS} (patience={PATIENCE})')
print(f'   Batch:   {BATCH_SIZE} | ImgSize: {IMG_SIZE}')

---
## 1️⃣ Montage Google Drive

In [ ]:
from google.colab import drive
import os

print('📂 Montage de Google Drive...')
drive.mount('/content/drive', force_remount=False)

# Vérification que les dossiers source existent
for path, name in [(RAW_IMAGES_DIR, 'Images'), (RAW_LABELS_DIR, 'Labels')]:
    if os.path.exists(path):
        count = len([f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))])
        print(f'  ✅ {name}: {path} ({count} fichiers)')
    else:
        print(f'  ❌ {name} INTROUVABLE: {path}')
        print(f'     → Vérifiez le chemin dans la Cellule 0')

# Créer le dossier de sortie Drive
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f'\n📁 Dossier de sortie: {DRIVE_OUTPUT_DIR}')

---
## 2️⃣ Installation des dépendances

In [ ]:
%%capture install_output

# Installation silencieuse des dépendances
!pip install ultralytics==8.3.0 --quiet
!pip install onnx onnxruntime --quiet
!pip install albumentations --quiet
!pip install matplotlib seaborn Pillow tqdm --quiet

In [ ]:
# --- Imports principaux ---
import os
import sys
import shutil
import random
import json
import yaml
import time
import glob
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image, ImageFile
from tqdm.notebook import tqdm

import torch
from ultralytics import YOLO

# Tolérance pour images tronquées/corrompues
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Fixer tous les seeds pour reproductibilité
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

import ultralytics
print(f'✅ Ultralytics YOLOv8: v{ultralytics.__version__}')
print(f'✅ PyTorch: v{torch.__version__}')
print(f'✅ Imports OK | Seed fixé: {RANDOM_SEED}')

---
## 3️⃣ Vérification GPU / CUDA

In [ ]:
print('=' * 55)
print('         🖥️  INFORMATIONS SYSTÈME')
print('=' * 55)

# CUDA
cuda_available = torch.cuda.is_available()
print(f'\n🔥 CUDA disponible : {cuda_available}')

if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    total_mem   = torch.cuda.get_device_properties(0).total_memory / 1e9
    free_mem    = (torch.cuda.get_device_properties(0).total_memory
                   - torch.cuda.memory_allocated(0)) / 1e9

    print(f'   GPU         : {device_name}')
    print(f'   VRAM totale : {total_mem:.1f} GB')
    print(f'   VRAM libre  : {free_mem:.1f} GB')
    print(f'   CUDA version: {torch.version.cuda}')
    DEVICE = 0
else:
    print('   ⚠️  Aucun GPU détecté — entraînement sur CPU (très lent !)')
    print('   → Allez dans Exécution > Modifier le type d\'exécution > GPU')
    DEVICE = 'cpu'

# RAM système
import psutil
ram = psutil.virtual_memory()
print(f'\n💾 RAM totale   : {ram.total / 1e9:.1f} GB')
print(f'   RAM libre    : {ram.available / 1e9:.1f} GB')

# Espace disque
disk = psutil.disk_usage('/content')
print(f'\n💿 Disque /content:')
print(f'   Total : {disk.total / 1e9:.1f} GB')
print(f'   Libre : {disk.free / 1e9:.1f} GB')

print('\n' + '=' * 55)

# Conseil batch size
if cuda_available and total_mem < 10:
    print(f'💡 VRAM < 10GB → BATCH_SIZE recommandé: 8-16')
elif cuda_available:
    print(f'💡 VRAM >= 10GB → BATCH_SIZE recommandé: 16-32')

---
## 4️⃣ Organisation du Dataset
### 4.1 — Copie locale & structure de dossiers

In [ ]:
# ============================================================
# ÉTAPE 4.1 — Copie du dataset depuis Drive vers /content
# (beaucoup plus rapide que de travailler depuis Drive)
# ============================================================

LOCAL_DATASET = '/content/dataset'

# Sous-dossiers finaux
dirs = {
    'images_train': f'{LOCAL_DATASET}/images/train',
    'images_val'  : f'{LOCAL_DATASET}/images/val',
    'labels_train': f'{LOCAL_DATASET}/labels/train',
    'labels_val'  : f'{LOCAL_DATASET}/labels/val',
}

# Nettoyage si déjà existant
if os.path.exists(LOCAL_DATASET):
    shutil.rmtree(LOCAL_DATASET)
    print('🗑️  Ancien dataset local supprimé')

for d in dirs.values():
    os.makedirs(d, exist_ok=True)

print('📁 Structure créée:')
print(f'''
{LOCAL_DATASET}/
 ├── images/
 │    ├── train/   ← {int((1-VAL_SPLIT)*100)}%
 │    └── val/     ← {int(VAL_SPLIT*100)}%
 ├── labels/
 │    ├── train/
 │    └── val/
''')

### 4.2 — Split train/val + copie des fichiers

In [ ]:
# ============================================================
# ÉTAPE 4.2 — Split train/val avec correspondance image↔label
# ============================================================

# Extensions d'images supportées
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

def get_image_files(directory):
    """Retourne la liste de toutes les images dans un dossier."""
    files = []
    for f in Path(directory).iterdir():
        if f.is_file() and f.suffix.lower() in IMG_EXTS:
            files.append(f)
    return sorted(files)

def find_label(img_path, labels_dir):
    """Cherche le label .txt correspondant à une image."""
    stem = img_path.stem
    label_path = Path(labels_dir) / f'{stem}.txt'
    return label_path if label_path.exists() else None

# Récupérer toutes les images
print('🔍 Scan du dataset source...')
all_images = get_image_files(RAW_IMAGES_DIR)
print(f'   Images trouvées: {len(all_images)}')

# Construire les paires image/label valides
valid_pairs  = []
missing_labels = []

for img in all_images:
    label = find_label(img, RAW_LABELS_DIR)
    if label is not None:
        valid_pairs.append((img, label))
    else:
        missing_labels.append(img.name)

print(f'   Paires valides (image+label): {len(valid_pairs)}')
if missing_labels:
    print(f'   ⚠️  {len(missing_labels)} images SANS label:')
    for m in missing_labels[:10]:
        print(f'      - {m}')
    if len(missing_labels) > 10:
        print(f'      ... et {len(missing_labels)-10} autres')

# Shuffle déterministe
random.shuffle(valid_pairs)

# Split
split_idx  = int(len(valid_pairs) * (1 - VAL_SPLIT))
train_pairs = valid_pairs[:split_idx]
val_pairs   = valid_pairs[split_idx:]

print(f'\n📊 Split effectué:')
print(f'   Train : {len(train_pairs)} paires ({len(train_pairs)/len(valid_pairs)*100:.1f}%)')
print(f'   Val   : {len(val_pairs)} paires ({len(val_pairs)/len(valid_pairs)*100:.1f}%)')

# Copie des fichiers
def copy_pairs(pairs, img_dest, lbl_dest, split_name):
    """Copie les paires image/label vers les dossiers destination."""
    for img_src, lbl_src in tqdm(pairs, desc=f'Copie {split_name}'):
        shutil.copy2(img_src, img_dest)
        shutil.copy2(lbl_src, lbl_dest)

copy_pairs(train_pairs, dirs['images_train'], dirs['labels_train'], 'train')
copy_pairs(val_pairs,   dirs['images_val'],   dirs['labels_val'],   'val  ')

print('\n✅ Dataset organisé avec succès!')

### 4.3 — Génération du fichier data.yaml

In [ ]:
# ============================================================
# ÉTAPE 4.3 — Génération automatique de data.yaml
# ============================================================

YAML_PATH = f'{LOCAL_DATASET}/data.yaml'

yaml_content = {
    'path'  : LOCAL_DATASET,
    'train' : 'images/train',
    'val'   : 'images/val',
    'nc'    : NUM_CLASSES,
    'names' : CLASS_NAMES,
}

with open(YAML_PATH, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False, allow_unicode=True)

print('📄 data.yaml généré:')
print('─' * 40)
with open(YAML_PATH) as f:
    print(f.read())
print('─' * 40)
print(f'✅ Sauvegardé: {YAML_PATH}')

---
## 5️⃣ Vérification de l'intégrité du Dataset

In [ ]:
# ============================================================
# ÉTAPE 5 — Vérification complète: images, labels, classes
# ============================================================

def verify_dataset(dataset_path, class_names):
    """Vérifie l'intégrité complète du dataset."""
    num_classes  = len(class_names)
    errors       = []
    warnings_list = []
    stats        = {'train': {}, 'val': {}}

    for split in ['train', 'val']:
        img_dir = Path(dataset_path) / 'images' / split
        lbl_dir = Path(dataset_path) / 'labels' / split

        images       = sorted(list(img_dir.glob('*.*')))
        corrupt_imgs = []
        missing_lbl  = []
        empty_lbl    = []
        invalid_cls  = []
        class_counts = {i: 0 for i in range(num_classes)}

        for img_path in images:
            if img_path.suffix.lower() not in IMG_EXTS:
                continue

            # 1. Vérification image
            try:
                img = Image.open(img_path)
                img.verify()
            except Exception as e:
                corrupt_imgs.append(img_path.name)
                continue

            # 2. Vérification label
            lbl_path = lbl_dir / f'{img_path.stem}.txt'
            if not lbl_path.exists():
                missing_lbl.append(img_path.name)
                continue

            # 3. Vérification contenu label
            with open(lbl_path) as f:
                lines = [l.strip() for l in f.readlines() if l.strip()]

            if not lines:
                empty_lbl.append(lbl_path.name)
                continue

            # 4. Vérification format et classes
            for line in lines:
                parts = line.split()
                if len(parts) != 5:
                    invalid_cls.append(f'{lbl_path.name}: mauvais format "{line}"')
                    continue
                try:
                    cls_id = int(parts[0])
                    coords = [float(x) for x in parts[1:]]
                    if cls_id < 0 or cls_id >= num_classes:
                        invalid_cls.append(f'{lbl_path.name}: classe {cls_id} invalide (max={num_classes-1})')
                    else:
                        class_counts[cls_id] += 1
                    # Vérifier que les coords sont dans [0,1]
                    if any(c < 0 or c > 1 for c in coords):
                        warnings_list.append(f'{lbl_path.name}: coordonnées hors [0,1]')
                except ValueError:
                    invalid_cls.append(f'{lbl_path.name}: valeur non-numérique')

        stats[split] = {
            'total'        : len(images),
            'corrupt'      : corrupt_imgs,
            'missing_label': missing_lbl,
            'empty_label'  : empty_lbl,
            'invalid_class': invalid_cls,
            'class_counts' : class_counts,
        }

    return stats, errors, warnings_list


print('🔬 Vérification du dataset...')
stats, errors, warns = verify_dataset(LOCAL_DATASET, CLASS_NAMES)

# Affichage du rapport
print('\n' + '=' * 55)
print('           📊 RAPPORT DE VÉRIFICATION')
print('=' * 55)

for split in ['train', 'val']:
    s = stats[split]
    ok = len(s['corrupt']) + len(s['missing_label']) + len(s['empty_label']) + len(s['invalid_class']) == 0
    icon = '✅' if ok else '⚠️ '
    print(f'\n{icon} [{split.upper()}] — {s["total"]} images')
    print(f'   Images corrompues  : {len(s["corrupt"])}')
    print(f'   Labels manquants   : {len(s["missing_label"])}')
    print(f'   Labels vides       : {len(s["empty_label"])}')
    print(f'   Classes invalides  : {len(s["invalid_class"])}')

    if s['invalid_class']:
        for msg in s['invalid_class'][:5]:
            print(f'      ⛔ {msg}')

    print(f'   Distribution classes:')
    for cls_id, count in s['class_counts'].items():
        bar = '█' * min(count // 5, 30)
        print(f'      [{cls_id:2d}] {CLASS_NAMES[cls_id]:12s}: {count:4d} {bar}')

if warns:
    print(f'\n⚠️  Avertissements ({len(warns)}):')
    for w in warns[:10]:
        print(f'   {w}')

print('\n' + '=' * 55)
print('✅ Vérification terminée')

### 5.1 — Visualisation de la distribution des classes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('📊 Distribution des classes — Train vs Val', fontsize=14, fontweight='bold')

colors = plt.cm.Set3(np.linspace(0, 1, NUM_CLASSES))

for ax, split in zip(axes, ['train', 'val']):
    counts = [stats[split]['class_counts'][i] for i in range(NUM_CLASSES)]
    bars = ax.bar(CLASS_NAMES, counts, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(f'{split.capitalize()} ({sum(counts)} annotations)', fontsize=12)
    ax.set_xlabel('Classes')
    ax.set_ylabel('Nombre d\'annotations')
    ax.tick_params(axis='x', rotation=45)
    # Valeur sur chaque barre
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(count), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'{DRIVE_OUTPUT_DIR}/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graphique sauvegardé dans Drive')

### 5.2 — Aperçu de quelques images avec annotations

In [ ]:
def yolo_to_xyxy(cx, cy, w, h, img_w, img_h):
    """Convertit les coords YOLO normalisées en pixels (x1,y1,x2,y2)."""
    x1 = (cx - w/2) * img_w
    y1 = (cy - h/2) * img_h
    x2 = (cx + w/2) * img_w
    y2 = (cy + h/2) * img_h
    return x1, y1, x2-x1, y2-y1  # x, y, width, height

def show_sample_images(img_dir, lbl_dir, class_names, n=8, title='Aperçu dataset'):
    """Affiche des images avec leurs bounding boxes."""
    img_paths = sorted(Path(img_dir).glob('*.*'))[:n]
    cols = 4
    rows = (len(img_paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    axes = axes.flatten() if rows > 1 else [axes] if len(img_paths) == 1 else axes.flatten()

    cmap = plt.cm.get_cmap('tab10', len(class_names))

    for i, img_path in enumerate(img_paths):
        ax = axes[i]
        try:
            img = np.array(Image.open(img_path).convert('RGB'))
            ax.imshow(img)
            h, w = img.shape[:2]

            lbl_path = Path(lbl_dir) / f'{img_path.stem}.txt'
            if lbl_path.exists():
                with open(lbl_path) as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) == 5:
                            cls_id = int(parts[0])
                            cx, cy, bw, bh = map(float, parts[1:])
                            x, y, bw_px, bh_px = yolo_to_xyxy(cx, cy, bw, bh, w, h)
                            color = cmap(cls_id % len(class_names))
                            rect = patches.Rectangle((x, y), bw_px, bh_px,
                                                     linewidth=2, edgecolor=color, facecolor='none')
                            ax.add_patch(rect)
                            ax.text(x, y - 4, class_names[cls_id] if cls_id < len(class_names) else f'id{cls_id}',
                                    color='white', fontsize=8, fontweight='bold',
                                    bbox=dict(boxstyle='round,pad=0.1', facecolor=color, alpha=0.8))
            ax.set_title(img_path.name[:25], fontsize=8)
        except Exception as e:
            ax.text(0.5, 0.5, f'Erreur:\n{e}', transform=ax.transAxes, ha='center')
        ax.axis('off')

    for j in range(i+1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.savefig(f'{DRIVE_OUTPUT_DIR}/sample_images.png', dpi=120, bbox_inches='tight')
    plt.show()

show_sample_images(
    dirs['images_train'], dirs['labels_train'],
    CLASS_NAMES, n=8, title='📸 Aperçu images train avec annotations'
)

---
## 6️⃣ Entraînement YOLOv8
### 6.1 — Chargement du modèle pré-entraîné

In [ ]:
# ============================================================
# ÉTAPE 6.1 — Chargement YOLOv8 pretrained
# ============================================================

if RESUME_TRAINING and RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT):
    print(f'▶️  Reprise depuis checkpoint: {RESUME_CHECKPOINT}')
    model = YOLO(RESUME_CHECKPOINT)
else:
    model_file = f'{MODEL_SIZE}.pt'
    print(f'⬇️  Téléchargement du modèle {model_file}...')
    model = YOLO(model_file)   # Téléchargement automatique depuis ultralytics
    print(f'✅ Modèle {model_file} chargé')

# Informations sur le modèle
print(f'\n📐 Architecture: {MODEL_SIZE}')
print(f'   Paramètres: {sum(p.numel() for p in model.model.parameters()):,}')
print(f'   Classes originales: {model.model.nc}')

### 6.2 — Lancement de l'entraînement

In [ ]:
# ============================================================
# ÉTAPE 6.2 — Entraînement avec tous les paramètres
# ============================================================

RUN_NAME = f'fruits_{MODEL_SIZE}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
PROJECT_DIR = '/content/runs/detect'

print('🚀 Démarrage de l\'entraînement...')
print(f'   Run name : {RUN_NAME}')
print(f'   Epochs   : {EPOCHS} (early stopping patience={PATIENCE})')
print(f'   Batch    : {BATCH_SIZE} | ImgSize: {IMG_SIZE}')
print(f'   Device   : {DEVICE}')
print(f'   Mixed FP : {USE_MIXED_PRECISION}')
print('─' * 55)

start_time = time.time()

results = model.train(
    # ── Dataset ──────────────────────────────────────────
    data       = YAML_PATH,
    imgsz      = IMG_SIZE,
    cache      = 'ram' if CACHE_DATASET else False,   # 'ram' ou 'disk'

    # ── Training ─────────────────────────────────────────
    epochs     = EPOCHS,
    batch      = BATCH_SIZE,
    device     = DEVICE,
    workers    = 2,                     # Colab: ne pas dépasser 4

    # ── Optimisation ─────────────────────────────────────
    optimizer  = OPTIMIZER,
    lr0        = LR0,
    lrf        = LRF,
    weight_decay = WEIGHT_DECAY,
    warmup_epochs= WARMUP_EPOCHS,
    warmup_momentum = 0.8,
    warmup_bias_lr  = 0.1,

    # ── Régularisation ───────────────────────────────────
    dropout    = 0.0,                   # Pas de dropout pour YOLO
    patience   = PATIENCE,              # Early stopping

    # ── Augmentation ─────────────────────────────────────
    **AUGMENT_PARAMS,

    # ── Précision ────────────────────────────────────────
    amp        = USE_MIXED_PRECISION,   # Automatic Mixed Precision (float16)

    # ── Sauvegarde ───────────────────────────────────────
    project    = PROJECT_DIR,
    name       = RUN_NAME,
    save       = True,
    save_period= 10,                    # Sauvegarder checkpoint tous les N epochs
    exist_ok   = True,

    # ── Logging ──────────────────────────────────────────
    plots      = True,                  # Générer les courbes
    verbose    = True,
    seed       = RANDOM_SEED,

    # ── Reprise ──────────────────────────────────────────
    resume     = RESUME_TRAINING,
)

elapsed = time.time() - start_time
print(f'\n✅ Entraînement terminé en {elapsed/60:.1f} minutes')

BEST_MODEL_PATH = f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt'
LAST_MODEL_PATH = f'{PROJECT_DIR}/{RUN_NAME}/weights/last.pt'
RUN_DIR = f'{PROJECT_DIR}/{RUN_NAME}'

print(f'   Meilleur modèle: {BEST_MODEL_PATH}')
print(f'   Dernier modèle : {LAST_MODEL_PATH}')

### 6.3 — Sauvegarde automatique dans Drive

In [ ]:
# ============================================================
# ÉTAPE 6.3 — Copie des résultats vers Google Drive
# ============================================================

drive_run_dir = os.path.join(DRIVE_OUTPUT_DIR, RUN_NAME)
os.makedirs(drive_run_dir, exist_ok=True)

print(f'📂 Sauvegarde dans Drive: {drive_run_dir}')

# 1. Poids du modèle
weights_src = f'{RUN_DIR}/weights'
weights_dst = f'{drive_run_dir}/weights'
if os.path.exists(weights_src):
    shutil.copytree(weights_src, weights_dst, dirs_exist_ok=True)
    print(f'  ✅ Poids copiés: {weights_dst}')

# 2. Fichiers de résultats (métriques, graphiques)
for ext in ['*.png', '*.jpg', '*.csv', '*.yaml']:
    for f in glob.glob(f'{RUN_DIR}/{ext}'):
        shutil.copy2(f, drive_run_dir)

print(f'  ✅ Graphiques & métriques copiés')

# 3. Copie du data.yaml
shutil.copy2(YAML_PATH, drive_run_dir)
print(f'  ✅ data.yaml copié')

# Résumé des fichiers sauvegardés
saved_files = os.listdir(drive_run_dir)
print(f'\n📋 Fichiers sauvegardés ({len(saved_files)}):')
for f in sorted(saved_files):
    size = os.path.getsize(os.path.join(drive_run_dir, f))
    print(f'   {f:<35} {size/1e6:.2f} MB')

---
## 7️⃣ Évaluation du modèle
### 7.1 — Métriques finales

In [ ]:
# ============================================================
# ÉTAPE 7.1 — Évaluation sur le set de validation
# ============================================================

print('📊 Évaluation du meilleur modèle...')
best_model = YOLO(BEST_MODEL_PATH)

val_results = best_model.val(
    data    = YAML_PATH,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    device  = DEVICE,
    verbose = True,
    plots   = True,
    save_json= True,
)

# Extraction des métriques principales
map50    = val_results.box.map50
map5095  = val_results.box.map
precision= val_results.box.mp
recall   = val_results.box.mr

print('\n' + '=' * 55)
print('         🏆 RÉSULTATS FINAUX')
print('=' * 55)
print(f'  mAP50      : {map50:.4f}  ({map50*100:.2f}%)')
print(f'  mAP50-95   : {map5095:.4f}  ({map5095*100:.2f}%)')
print(f'  Précision  : {precision:.4f}  ({precision*100:.2f}%)')
print(f'  Recall     : {recall:.4f}  ({recall*100:.2f}%)')
f1_score = 2 * precision * recall / (precision + recall + 1e-8)
print(f'  F1-Score   : {f1_score:.4f}  ({f1_score*100:.2f}%)')
print('=' * 55)

# Interprétation
if map50 >= 0.80:
    print('\n🟢 Excellent ! mAP50 >= 80%')
elif map50 >= 0.60:
    print('\n🟡 Bon résultat. mAP50 >= 60%')
elif map50 >= 0.40:
    print('\n🟠 Résultat moyen. Essayez plus d\'epochs ou un modèle plus grand.')
else:
    print('\n🔴 mAP50 < 40%. Vérifiez la qualité des annotations et le dataset.')

### 7.2 — Courbes d'entraînement

In [ ]:
# ============================================================
# ÉTAPE 7.2 — Affichage des courbes d'entraînement
# ============================================================

import pandas as pd

results_csv = f'{RUN_DIR}/results.csv'

if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()   # Nettoyer les espaces

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'📈 Courbes d\'entraînement — {RUN_NAME}', fontsize=14, fontweight='bold')

    # Mapping colonnes → titre
    plots_config = [
        ('train/box_loss',  'val/box_loss',  'Loss BBox',    'Loss'),
        ('train/cls_loss',  'val/cls_loss',  'Loss Classes', 'Loss'),
        ('train/dfl_loss',  'val/dfl_loss',  'Loss DFL',     'Loss'),
        ('metrics/precision(B)', None,          'Précision',    'Score'),
        ('metrics/recall(B)',    None,          'Recall',       'Score'),
        ('metrics/mAP50(B)',     None,          'mAP50',        'Score'),
    ]

    for ax, (train_col, val_col, title, ylabel) in zip(axes.flatten(), plots_config):
        if train_col in df.columns:
            ax.plot(df[train_col], label='Train', color='#2196F3', linewidth=2)
        if val_col and val_col in df.columns:
            ax.plot(df[val_col], label='Val', color='#F44336', linewidth=2, linestyle='--')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0)

    plt.tight_layout()
    curve_path = f'{drive_run_dir}/training_curves.png'
    plt.savefig(curve_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Courbes sauvegardées: {curve_path}')
else:
    print('⚠️  Fichier results.csv introuvable. Affichage du graphique généré par YOLO:')
    results_img = f'{RUN_DIR}/results.png'
    if os.path.exists(results_img):
        img = Image.open(results_img)
        plt.figure(figsize=(16, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.show()

### 7.3 — Prédictions sur des images de validation

In [ ]:
# ============================================================
# ÉTAPE 7.3 — Visualisation des prédictions
# ============================================================

CONF_THRESHOLD = 0.25   # Seuil de confiance
IOU_THRESHOLD  = 0.45   # Seuil IoU NMS
NUM_PREVIEW    = 8      # Nombre d'images à afficher

val_images = sorted(Path(dirs['images_val']).glob('*.*'))[:NUM_PREVIEW]

predictions = best_model.predict(
    source = [str(p) for p in val_images],
    conf   = CONF_THRESHOLD,
    iou    = IOU_THRESHOLD,
    device = DEVICE,
    verbose= False,
)

cols = 4
rows = (NUM_PREVIEW + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
fig.suptitle(f'🔍 Prédictions sur validation (conf≥{CONF_THRESHOLD})', fontsize=14, fontweight='bold')
axes = axes.flatten()

cmap = plt.cm.get_cmap('tab10', NUM_CLASSES)

for i, (result, img_path) in enumerate(zip(predictions, val_images)):
    ax = axes[i]
    img = np.array(Image.open(img_path).convert('RGB'))
    ax.imshow(img)

    boxes  = result.boxes
    n_det  = len(boxes) if boxes is not None else 0

    if n_det > 0:
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            cls_id  = int(box.cls[0])
            conf    = float(box.conf[0])
            color   = cmap(cls_id % NUM_CLASSES)
            name    = CLASS_NAMES[cls_id] if cls_id < NUM_CLASSES else f'id{cls_id}'

            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                      linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1 - 5, f'{name} {conf:.2f}',
                    color='white', fontsize=8, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.1', facecolor=color, alpha=0.85))

    ax.set_title(f'{img_path.name[:20]} ({n_det} det.)', fontsize=8)
    ax.axis('off')

for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
pred_path = f'{drive_run_dir}/predictions_val.png'
plt.savefig(pred_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Prédictions sauvegardées: {pred_path}')

### 7.4 — Matrice de confusion et métriques par classe

In [ ]:
# ============================================================
# ÉTAPE 7.4 — Affichage de la matrice de confusion YOLO
# ============================================================

# YOLO génère automatiquement la confusion matrix lors de val(plots=True)
confusion_matrix_path = f'{RUN_DIR}/confusion_matrix.png'
conf_norm_path = f'{RUN_DIR}/confusion_matrix_normalized.png'

for path, title in [(confusion_matrix_path, 'Matrice de confusion'),
                    (conf_norm_path, 'Matrice de confusion normalisée')]:
    if os.path.exists(path):
        img = Image.open(path)
        plt.figure(figsize=(12, 10))
        plt.imshow(img)
        plt.title(title, fontsize=14, fontweight='bold', pad=15)
        plt.axis('off')
        shutil.copy2(path, drive_run_dir)
        plt.show()
        print(f'✅ {title} sauvegardée dans Drive')
    else:
        print(f'⚠️  {title} non trouvée (générée uniquement si val(plots=True))')

# Métriques par classe (si disponibles)
pr_curve = f'{RUN_DIR}/PR_curve.png'
f1_curve = f'{RUN_DIR}/F1_curve.png'

for path, title in [(pr_curve, 'Courbe PR'), (f1_curve, 'Courbe F1')]:
    if os.path.exists(path):
        img = Image.open(path)
        plt.figure(figsize=(10, 7))
        plt.imshow(img)
        plt.title(title, fontsize=12, fontweight='bold')
        plt.axis('off')
        shutil.copy2(path, drive_run_dir)
        plt.show()

---
## 8️⃣ Export ONNX

In [ ]:
# ============================================================
# ÉTAPE 8 — Export du modèle en format ONNX
# (compatible avec OpenCV, TensorRT, ONNX Runtime, etc.)
# ============================================================

print('📦 Export ONNX...')

onnx_path = best_model.export(
    format   = 'onnx',
    imgsz    = IMG_SIZE,
    dynamic  = True,       # Batch size dynamique
    simplify = True,       # Simplification du graphe
    opset    = 17,         # Version ONNX opset
    half     = False,      # FP32 pour compatibilité max
)

print(f'✅ Modèle ONNX exporté: {onnx_path}')

# Copie vers Drive
if os.path.exists(str(onnx_path)):
    onnx_drive_path = os.path.join(drive_run_dir, 'weights', 'best.onnx')
    os.makedirs(os.path.dirname(onnx_drive_path), exist_ok=True)
    shutil.copy2(str(onnx_path), onnx_drive_path)
    size_mb = os.path.getsize(onnx_drive_path) / 1e6
    print(f'   Copié dans Drive: {onnx_drive_path} ({size_mb:.1f} MB)')

# Vérification rapide ONNX
try:
    import onnx
    model_onnx = onnx.load(str(onnx_path))
    onnx.checker.check_model(model_onnx)
    print('✅ Modèle ONNX valide (checker OK)')
except Exception as e:
    print(f'⚠️  Erreur vérification ONNX: {e}')

---
## 9️⃣ Résumé final & Fichier de configuration sauvegardé

In [ ]:
# ============================================================
# ÉTAPE 9 — Résumé complet + sauvegarde config
# ============================================================

# Sauvegarder la configuration complète
config_summary = {
    'run_name'       : RUN_NAME,
    'date'           : datetime.now().isoformat(),
    'model'          : MODEL_SIZE,
    'num_classes'    : NUM_CLASSES,
    'class_names'    : CLASS_NAMES,
    'dataset': {
        'train_samples': len(train_pairs),
        'val_samples'  : len(val_pairs),
        'val_split'    : VAL_SPLIT,
    },
    'training': {
        'epochs'      : EPOCHS,
        'batch_size'  : BATCH_SIZE,
        'img_size'    : IMG_SIZE,
        'optimizer'   : OPTIMIZER,
        'lr0'         : LR0,
        'lrf'         : LRF,
        'patience'    : PATIENCE,
        'weight_decay': WEIGHT_DECAY,
        'amp'         : USE_MIXED_PRECISION,
        'seed'        : RANDOM_SEED,
    },
    'results': {
        'mAP50'     : round(float(map50), 4),
        'mAP50_95'  : round(float(map5095), 4),
        'precision' : round(float(precision), 4),
        'recall'    : round(float(recall), 4),
        'f1_score'  : round(float(f1_score), 4),
    },
    'paths': {
        'best_model' : BEST_MODEL_PATH,
        'last_model' : LAST_MODEL_PATH,
        'drive_output': drive_run_dir,
    }
}

config_path = os.path.join(drive_run_dir, 'run_config.json')
with open(config_path, 'w') as f:
    json.dump(config_summary, f, indent=2, ensure_ascii=False)

# Affichage du résumé final
print('\n' + '╔' + '═'*53 + '╗')
print('║' + '         ✅ ENTRAÎNEMENT TERMINÉ'.center(53) + '║')
print('╠' + '═'*53 + '╣')
print(f'║  Run        : {RUN_NAME:<38}║')
print(f'║  Modèle     : {MODEL_SIZE:<38}║')
print(f'║  Classes    : {NUM_CLASSES:<38}║')
print(f'║  Train imgs : {len(train_pairs):<38}║')
print(f'║  Val imgs   : {len(val_pairs):<38}║')
print('╠' + '═'*53 + '╣')
print(f'║  mAP50      : {map50:.4f}  ({map50*100:.2f}%)          ║')
print(f'║  mAP50-95   : {map5095:.4f}  ({map5095*100:.2f}%)          ║')
print(f'║  Précision  : {precision:.4f}  ({precision*100:.2f}%)          ║')
print(f'║  Recall     : {recall:.4f}  ({recall*100:.2f}%)          ║')
print(f'║  F1-Score   : {f1_score:.4f}  ({f1_score*100:.2f}%)          ║')
print('╠' + '═'*53 + '╣')
print(f'║  Sauvegardé : Drive/{RUN_NAME[:34]}║')
print('╚' + '═'*53 + '╝')
print(f'\n📄 Config complète: {config_path}')

---
## 🔟 (Optionnel) Test sur une image personnalisée

In [ ]:
# ============================================================
# OPTIONNEL — Tester le modèle sur une image de votre choix
# ============================================================

# 👇 Modifier ce chemin pour tester votre propre image
TEST_IMAGE_PATH = ''   # Ex: '/content/drive/MyDrive/mes_images/test.jpg'

if TEST_IMAGE_PATH and os.path.exists(TEST_IMAGE_PATH):
    result = best_model.predict(
        source = TEST_IMAGE_PATH,
        conf   = 0.25,
        iou    = 0.45,
        device = DEVICE,
        verbose= False,
    )[0]

    img = np.array(Image.open(TEST_IMAGE_PATH).convert('RGB'))
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(img)

    if result.boxes is not None and len(result.boxes) > 0:
        cmap = plt.cm.get_cmap('tab10', NUM_CLASSES)
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            cls_id = int(box.cls[0])
            conf   = float(box.conf[0])
            color  = cmap(cls_id % NUM_CLASSES)
            name   = CLASS_NAMES[cls_id] if cls_id < NUM_CLASSES else f'id{cls_id}'

            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                      linewidth=3, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1 - 8, f'{name} {conf:.2f}',
                    color='white', fontsize=11, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.85))

        print(f'✅ {len(result.boxes)} objet(s) détecté(s)')
    else:
        print('⚠️  Aucun objet détecté (essayez de baisser conf)')

    ax.set_title(os.path.basename(TEST_IMAGE_PATH), fontsize=12)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

else:
    print('ℹ️  Renseignez TEST_IMAGE_PATH pour tester sur une image personnalisée.')
    print('   Exemple:')
    print('   TEST_IMAGE_PATH = "/content/drive/MyDrive/mes_images/test.jpg"')

---
## 📌 Guide de réutilisation

### Charger le modèle pour l'inférence
```python
from ultralytics import YOLO

# Charger depuis Drive
model = YOLO('/content/drive/MyDrive/yolov8_fruits_runs/<run_name>/weights/best.pt')

# Inférence sur une image
results = model.predict('image.jpg', conf=0.25)
results[0].show()   # Afficher
results[0].save()   # Sauvegarder
```

### Reprendre un entraînement interrompu
```python
# Dans la Cellule 0, modifier:
RESUME_TRAINING   = True
RESUME_CHECKPOINT = '/content/drive/MyDrive/yolov8_fruits_runs/<run_name>/weights/last.pt'
```

### Utiliser un modèle plus grand
```python
# Dans la Cellule 0, changer:
MODEL_SIZE = 'yolov8s'   # small  (~11M params)
MODEL_SIZE = 'yolov8m'   # medium (~26M params)
MODEL_SIZE = 'yolov8l'   # large  (~44M params)
```

### Classes et IDs
| Index YOLO | Nom | ID COCO original |
|---|---|---|
| 0 | melon | 80 |
| 1 | grape | 81 |
| 2 | peach | 82 |
| 3 | avocado | 83 |
| 4 | pineapple | 84 |
| 5 | tomato | 85 |
| 6 | strawberry | 86 |
| 7 | mango | 87 |
| 8 | pear | 88 |